# ARTI5001 Assignment 2 - FFHQ128 Variational Autoencoder

**Student name:**  
**Student ID:**  

Submit this notebook as `<student_id>_<name>_A2.ipynb` after restarting the
kernel and running all cells in order. Keep all required outputs visible.

This notebook provides data-loading infrastructure and evidence headings. It
does not prescribe the assessed model, loss, training loop, or analysis.


## Assessment conditions

- Implement the convolutional VAE, reparameterisation, analytic KL, objective,
  and training/evaluation logic yourself.
- Train the fixed baseline and exactly one approved controlled variant.
- Use the public rubric supplied with the assignment.
- Checkpoint saving is optional and ungraded.
- Complete the mandatory source and AI-use declaration at the end.
- Do not upload FFHQ images or protected course material to external AI tools.


## 1. Reproducible setup and data checks

Configure the experiment in one place. Both runs must use the same seed, split,
batch size, optimiser, learning rate, epoch count, and evaluation procedure.


In [ ]:
from pathlib import Path
import copy
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid

DATA_ROOT = Path("./ffhq128")
MANIFEST_PATH = DATA_ROOT / "ffhq128_split_manifest.csv"
IMAGE_ROOT = DATA_ROOT / "images"

SEED = 5001
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 15
BASELINE_LATENT_DIM = 16
BASELINE_BETA = 1.0
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
# Supplied infrastructure: do not modify this cell.
class ManifestFFHQ128(Dataset):
    def __init__(self, image_root, manifest_path, split):
        table = pd.read_csv(manifest_path)
        required = {"image_id", "filename", "split"}
        if not required.issubset(table.columns):
            raise ValueError(f"Manifest must contain {sorted(required)}")
        self.rows = table.loc[table["split"] == split].reset_index(drop=True)
        self.image_root = Path(image_root)
        self.to_tensor = ToTensor()

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        path = self.image_root / row["filename"]
        with Image.open(path) as image:
            tensor = self.to_tensor(image.convert("RGB"))
        return tensor, int(row["image_id"])


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_loaders():
    datasets = {
        split: ManifestFFHQ128(IMAGE_ROOT, MANIFEST_PATH, split)
        for split in ("train", "val", "test")
    }
    generator = torch.Generator().manual_seed(SEED)
    loaders = {
        "train": DataLoader(datasets["train"], batch_size=BATCH_SIZE,
                            shuffle=True, generator=generator,
                            num_workers=NUM_WORKERS, pin_memory=True),
        "val": DataLoader(datasets["val"], batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=True),
        "test": DataLoader(datasets["test"], batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=NUM_WORKERS,
                           pin_memory=True),
    }
    return datasets, loaders


set_seed(SEED)
datasets, loaders = make_loaders()


In [ ]:
# Show split sizes, batch shape/range, sample images, and any other
# checks needed to make the pipeline and experiment auditable.



## 2. Convolutional VAE architecture

Implement a convolutional encoder and decoder for `3 x 128 x 128` images. The
forward pass must return the reconstruction or logits, `mu`, and `logvar`.
Record the trainable-parameter count and keep it at or below 10 million.

Explain tensor shapes and key architecture decisions in markdown near the code.


In [ ]:
# Implement your convolutional VAE here.



In [ ]:
# Instantiate a baseline model and display shape and parameter-count checks.



## 3. VAE objective and verification

Implement a reconstruction term, analytic diagonal-Gaussian KL against
`N(0,I)`, and `reconstruction + beta * KL`. Use and document a coherent
per-image reduction. Report unweighted KL separately.


In [ ]:
# Implement the reconstruction term, KL term, and total objective here.



In [ ]:
# Run shape, finite-loss, and KL non-negativity checks here.



## 4. Training and validation

For every epoch, retain train and validation total, reconstruction, and
unweighted KL values. Both baseline and variant must run for 15 epochs under
the same declared controls.


In [ ]:
# Implement reusable training and evaluation functions here.
# The implementation must make the history for all six required series easy to audit.



### 4.1 Baseline run

Baseline condition: `latent_dim=16`, fixed `beta=1.0`, 15 epochs, no data
augmentation. State optimiser and learning-rate rationale before training.


In [ ]:
# Reset the seed, recreate the loaders, then instantiate, train, and
# validate the baseline here. Recreating loaders resets the shuffle generator.



### 4.2 Controlled variant

Select exactly one:

- latent dimension 16 to 32;
- fixed beta 1.0 to 0.25 or 2.0;
- fixed beta 1.0 to linear warm-up from 0 to 1 over the first 30% of epochs.

**Selected factor:**  
**Prediction before examining results:**  
**Controls held constant:**  


In [ ]:
# Reset the same seed, recreate the loaders, then instantiate, train, and
# validate the variant here. Recreating loaders resets the shuffle generator.



## 5. Required evaluation evidence

Use the same test examples, interpolation endpoints, and generation seed for
both models. Clearly identify baseline and variant in every table and figure.


### 5.1 Quantitative test comparison

Report mean per-image reconstruction, unweighted KL, and total objective for
both models.


In [ ]:
# Create and display the baseline-versus-variant test result table here.



### 5.2 Reconstruction comparison

Show the same 12 test images as original, baseline reconstruction, and variant
reconstruction.


In [ ]:
# Produce the required reconstruction comparison here.



### 5.3 Prior samples

Show at least 64 samples from each model in an 8 by 8 grid, using the same
generation seed.


In [ ]:
# Produce the baseline and variant prior-sample grids here.



### 5.4 Latent interpolation

Use at least 12 interpolation steps between the same two encoded test images
for both models.


In [ ]:
# Produce both latent interpolations here.



### 5.5 Training and validation curves

Plot total, reconstruction, and unweighted KL for train and validation for both
models. Labels must distinguish model, split, and loss component.


In [ ]:
# Produce the required comparison curves here.



## 6. Notebook report

### A. Design rationale

Explain the architecture, reconstruction term, decoder output, optimiser,
learning rate, and stability choices. Tie each choice to FFHQ128 and your model.

**Response:**



### B. Controlled comparison

Evaluate your prediction using at least three exact own-run values and at least
three specific figure observations. Explain the reconstruction-KL trade-off and
identify at least one visible failure case.

**Response:**



### C. Limitations and next experiment

Identify limitations visible in your results. Propose one technically specific,
testable next experiment and state what outcome would support it.

**Response:**



## 7. References, sources, and AI-use declaration

This section is mandatory. Add one row for every external source or AI tool.
Identify exact affected cells, functions, figures, or report sections. Explain
what was retained or adapted and how you independently verified it.

| Source/tool and version | Purpose | Exact affected cells/functions/sections | Material retained or adapted | Independent verification |
|---|---|---|---|---|
| None / add source |  |  |  |  |

**Generative AI declaration:** Used / Not used  
**External code or tutorial declaration:** Used / Not used  

References:



## Final submission check

- [ ] Baseline and one variant each have 15 visible epochs.
- [ ] Train/validation total, reconstruction, and unweighted KL are visible.
- [ ] Test result table and all required figures are visible.
- [ ] Numerical and figure-specific report claims match notebook evidence.
- [ ] Source and AI-use declaration is complete and specific.
- [ ] Kernel restart and Run All completes without unresolved errors.
